# H3 Motion on Google Colab

Run the H3 Motion web app and its ComfyUI inference server in one Colab GPU runtime. The app is served through a Cloudflare Quick Tunnel; ComfyUI and its API proxy stay on the runtime's local network.

This notebook pins the ComfyUI and ComfyUI-vlo revisions used by the project, downloads the model files listed in the [H3 Motion setup guide](../README.md#setup), builds the existing frontend, and starts the app.

## Before you start

- In Colab, choose **Runtime → Change runtime type → GPU**.
- Add your Hugging Face access token as a Colab Secret named `HF_TOKEN`, and enable notebook access. The token is read from Secrets and never printed.
- Confirm your Hugging Face account has access to each model repository and that you have accepted its terms.
- Run the cells from top to bottom. Colab storage is temporary; a new runtime requires setup and downloads again.

The app's model inventory includes both H3 routes so its environment check can pass. **PX-706 Draw-to-Animate uses the photo / fl2va route.** The ref2va files enable the existing video-inpainting route; this notebook does not develop that feature.

## 1. Check the runtime and set paths

In [ ]:
import json
import os
import platform
import re
import secrets
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import Request, urlopen

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab, then rerun this cell.")

BASE_DIR = Path("/content/h3_motion_colab")
COMFYUI_DIR = BASE_DIR / "ComfyUI"
VLO_DIR = COMFYUI_DIR / "custom_nodes" / "ComfyUI-vlo"
APP_DIR = BASE_DIR / "h3-motion"
MODELS_DIR = COMFYUI_DIR / "models"

COMFYUI_REPO = "https://github.com/Comfy-Org/ComfyUI.git"
COMFYUI_COMMIT = "6607f36b53c7b695b160ac5b52d595e9e3752a2d"
VLO_REPO = "https://github.com/PxTicks/ComfyUI-vlo.git"
VLO_COMMIT = "668db1d123bd4de71edce71ad40a4d2e97633a79"
APP_REPO = "https://github.com/Yuvrajxms09/h3-motion-preview-download.git"
APP_BRANCH = "main"

COMFY_URL = "http://127.0.0.1:8188"
PROXY_URL = "http://127.0.0.1:8189"
APP_URL = "http://127.0.0.1:7860"

BASE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSES = globals().get("PROCESSES", {})
LOG_HANDLES = globals().get("LOG_HANDLES", {})
ACCESS_KEY = globals().get("ACCESS_KEY") or secrets.token_urlsafe(32)
PUBLIC_URL = globals().get("PUBLIC_URL")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"Free runtime disk: {shutil.disk_usage(BASE_DIR).free / 1024**3:.1f} GiB")
print(f"Workspace: {BASE_DIR}")

## 2. Install the pinned ComfyUI revision

This uses the tested project commit. It leaves Colab's preinstalled PyTorch in place and installs ComfyUI's own Python requirements.

In [ ]:
def run(command, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)


def ensure_checkout(path, repo_url, commit):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", repo_url, str(path)])
    elif not (path / ".git").is_dir():
        raise RuntimeError(f"Refusing to replace non-Git directory: {path}")

    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=path, text=True
    ).strip().removesuffix(".git").rstrip("/")
    expected = repo_url.removesuffix(".git").rstrip("/")
    if origin != expected:
        raise RuntimeError(f"Unexpected origin for {path}: {origin}")

    run(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    run(["git", "checkout", "--detach", commit], cwd=path)


ensure_checkout(COMFYUI_DIR, COMFYUI_REPO, COMFYUI_COMMIT)
# Colab supplies matching CUDA builds of torch and torchvision; keep those in place.
requirements_path = COMFYUI_DIR / "requirements.txt"
requirements_for_colab = BASE_DIR / "comfyui-colab-requirements.txt"
requirements_for_colab.write_text(
    "\n".join(
        line for line in requirements_path.read_text().splitlines()
        if line.strip().lower() not in {"torch", "torchvision"}
    ) + "\n",
    encoding="utf-8",
)
run([sys.executable, "-m", "pip", "install", "-r", str(requirements_for_colab)])

installed_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=COMFYUI_DIR, text=True
).strip()
if installed_commit != COMFYUI_COMMIT:
    raise RuntimeError(f"ComfyUI commit mismatch: {installed_commit}")
print(f"ComfyUI ready at {COMFYUI_COMMIT}")

## 3. Install ComfyUI-vlo

H3 Motion uses this pack for SAM refinement, mask handling, and `vloTimeToMove`. No other custom-node packs are required by the project workflows.

In [ ]:
ensure_checkout(VLO_DIR, VLO_REPO, VLO_COMMIT)

vlo_requirements = VLO_DIR / "requirements.txt"
if vlo_requirements.is_file():
    run([sys.executable, "-m", "pip", "install", "-r", str(vlo_requirements)])

installed_vlo_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=VLO_DIR, text=True
).strip()
if installed_vlo_commit != VLO_COMMIT:
    raise RuntimeError(f"ComfyUI-vlo commit mismatch: {installed_vlo_commit}")
print(f"ComfyUI-vlo ready at {VLO_COMMIT}")

## 4. Download the app's model files

Files go into the exact ComfyUI model folders and use the repository revisions in H3 Motion's setup guide. Existing non-empty files are reused; interrupted downloads can be rerun safely.

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "Colab Secret 'HF_TOKEN' is missing or notebook access is disabled."
    )

MODEL_FILES = [
    # SAM 3.1 subject selection
    ("Comfy-Org/sam3.1", "checkpoints/sam3.1_multiplex_fp16.safetensors", "7bb8374780a725b4353ed31f3a9395c9742b5621"),
    # Qwen Image 2.1: image description and background-plate editing
    ("Comfy-Org/Qwen-Image-2.1", "diffusion_models/qwen_image_2.1_int8_convrot.safetensors", "0217efd1d9c9452e84f6448903d6e21bfa171ffd"),
    ("Comfy-Org/Qwen-Image-2.1", "text_encoders/qwen3vl_8b_int8_convrot.safetensors", "0217efd1d9c9452e84f6448903d6e21bfa171ffd"),
    ("Comfy-Org/Qwen-Image-2.1", "vae/qwen_image_2.1_vae_bf16.safetensors", "0217efd1d9c9452e84f6448903d6e21bfa171ffd"),
    # H3 photo animation and video inpainting routes
    ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "912b6e38b2092ec4d2e86d24183e0d59ea059d48"),
    ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", "912b6e38b2092ec4d2e86d24183e0d59ea059d48"),
    ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "912b6e38b2092ec4d2e86d24183e0d59ea059d48"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "912b6e38b2092ec4d2e86d24183e0d59ea059d48"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "912b6e38b2092ec4d2e86d24183e0d59ea059d48"),
    ("Comfy-Org/MiniMax-H3", "loras/pika_h3_flex_ref2va_v1_bf16.safetensors", "912b6e38b2092ec4d2e86d24183e0d59ea059d48"),
]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
for repo_id, filename, revision in MODEL_FILES:
    destination = MODELS_DIR / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and destination.stat().st_size > 0:
        print(f"Already present: {filename}")
        continue

    print(f"Downloading {repo_id}/{filename}")
    try:
        downloaded = Path(
            hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                revision=revision,
                token=HF_TOKEN,
                local_dir=str(MODELS_DIR),
            )
        )
    except Exception as error:
        raise RuntimeError(
            f"Could not download {repo_id}/{filename}. Check HF_TOKEN and your access to that model repository."
        ) from error

    if not downloaded.is_file() or downloaded.stat().st_size == 0:
        raise RuntimeError(f"Download is missing or empty: {downloaded}")
    print(f"Ready: {filename} ({downloaded.stat().st_size / 1024**3:.2f} GiB)")

missing = [filename for _, filename, _ in MODEL_FILES if not (MODELS_DIR / filename).is_file()]
if missing:
    raise RuntimeError("Missing model files: " + ", ".join(missing))
print(f"All {len(MODEL_FILES)} model files are present.")

## 5. Install Node.js and build H3 Motion

The app requires Node.js 22.12+ (or 24+). If Colab's installed Node version is older, this cell installs the Node.js 22.x LTS package, then clones and builds the app.

In [ ]:
def node_version():
    if not shutil.which("node"):
        return None
    output = subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v")
    major, minor, *_ = (int(part) for part in output.split("."))
    return major, minor, output


def supported_node_version(version):
    return version is not None and (
        (version[0] == 22 and version[1] >= 12) or version[0] >= 24
    )

version = node_version()
if not supported_node_version(version):
    subprocess.run(
        "curl -fsSL https://deb.nodesource.com/setup_22.x | bash -",
        shell=True,
        executable="/bin/bash",
        check=True,
    )
    run(["apt-get", "install", "-y", "nodejs"])
    version = node_version()

if not supported_node_version(version):
    raise RuntimeError(f"Node.js 22.12+ or 24+ is required; found {version}")
print(f"Node.js {version[2]}; npm {subprocess.check_output(['npm', '--version'], text=True).strip()}")

if not APP_DIR.exists():
    run(["git", "clone", "--depth", "1", "--branch", APP_BRANCH, APP_REPO, str(APP_DIR)])
elif not (APP_DIR / ".git").is_dir():
    raise RuntimeError(f"Refusing to replace non-Git directory: {APP_DIR}")
else:
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=APP_DIR, text=True
    ).strip().removesuffix(".git").rstrip("/")
    if origin != APP_REPO.removesuffix(".git"):
        raise RuntimeError(f"Unexpected H3 Motion repository origin: {origin}")

run([sys.executable, "-m", "pip", "install", "comfy-api-proxy==0.1.6"])
run(["npm", "ci"], cwd=APP_DIR)
run(["npm", "run", "build"], cwd=APP_DIR)

app_commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=APP_DIR, text=True
).strip()
print(f"H3 Motion built from {app_commit}")

## 6. Install Cloudflare Tunnel

The tunnel will expose the H3 Motion app only. The ComfyUI UI and API are not exposed publicly.

In [ ]:
import urllib.request

if platform.machine() != "x86_64":
    raise RuntimeError(f"This Colab installer expects x86_64; found {platform.machine()}.")

if not shutil.which("cloudflared"):
    deb_path = BASE_DIR / "cloudflared-linux-amd64.deb"
    deb_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
    print("Downloading cloudflared.")
    urllib.request.urlretrieve(deb_url, deb_path)
    run(["dpkg", "-i", str(deb_path)])

if not shutil.which("cloudflared"):
    raise RuntimeError("cloudflared installation did not produce an executable.")
print(subprocess.check_output(["cloudflared", "--version"], text=True).strip())

## 7. Start ComfyUI

ComfyUI listens on loopback port 8188. The model and custom-node setup is complete before it starts, so no manual ComfyUI restart is needed.

In [ ]:
def read_json(url, timeout=5, headers=None):
    request = Request(url, headers=headers or {})
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)


def log_tail(path, lines=50):
    path = Path(path)
    if not path.is_file():
        return "(log file not found)"
    return "\n".join(path.read_text(errors="replace").splitlines()[-lines:])


def wait_for_json(url, name, timeout=300, process=None, log_path=None, headers=None):
    deadline = time.monotonic() + timeout
    last_error = None
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            details = f"\n{log_tail(log_path)}" if log_path else ""
            raise RuntimeError(f"{name} exited during startup.{details}")
        try:
            return read_json(url, headers=headers)
        except Exception as error:
            last_error = error
            time.sleep(2)
    details = f"\n{log_tail(log_path)}" if log_path else ""
    raise TimeoutError(f"Timed out waiting for {name}: {last_error}{details}")


comfy_log = BASE_DIR / "comfyui.log"
comfy_process = PROCESSES.get("comfy")
try:
    comfy_status = read_json(f"{COMFY_URL}/system_stats")
except Exception:
    if comfy_process is not None and comfy_process.poll() is None:
        raise RuntimeError(f"ComfyUI is running but not responding.\n{log_tail(comfy_log)}")
    handle = comfy_log.open("w")
    LOG_HANDLES["comfy"] = handle
    PROCESSES["comfy"] = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=COMFYUI_DIR,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    comfy_status = wait_for_json(
        f"{COMFY_URL}/system_stats", "ComfyUI", timeout=600,
        process=PROCESSES["comfy"], log_path=comfy_log,
    )
else:
    if comfy_process is None or comfy_process.poll() is not None:
        raise RuntimeError(
            "Port 8188 is already serving a ComfyUI outside this notebook. "
            "Use a fresh Colab runtime so this setup can use its pinned checkout."
        )
    print("Reusing this notebook's ComfyUI server.")

print(f"ComfyUI is responding at {COMFY_URL}")

## 8. Start the API proxy and check nodes/models

The proxy connects the H3 Motion server to ComfyUI's v2 API. The project's existing environment checker then confirms the expected node classes and model filenames are visible to ComfyUI.

In [ ]:
proxy_log = BASE_DIR / "comfy-api-proxy.log"
proxy_process = PROCESSES.get("proxy")
try:
    proxy_status = read_json(f"{PROXY_URL}/api/v2/health")
except Exception:
    if proxy_process is not None and proxy_process.poll() is None:
        raise RuntimeError(f"The API proxy is running but not responding.\n{log_tail(proxy_log)}")
    handle = proxy_log.open("w")
    LOG_HANDLES["proxy"] = handle
    PROCESSES["proxy"] = subprocess.Popen(
        [
            "comfy-api-proxy",
            "--comfyui", COMFY_URL,
            "--host", "127.0.0.1",
            "--port", "8189",
            "--max-upload-mb", "500",
        ],
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    proxy_status = wait_for_json(
        f"{PROXY_URL}/api/v2/health", "Comfy API proxy", timeout=120,
        process=PROCESSES["proxy"], log_path=proxy_log,
    )
else:
    if proxy_process is None or proxy_process.poll() is not None:
        raise RuntimeError(
            "Port 8189 is already serving an API proxy outside this notebook. "
            "Use a fresh Colab runtime so the proxy can point to this ComfyUI."
        )
    print("Reusing this notebook's Comfy API proxy.")

if proxy_status.get("status") not in {"healthy", "ok"}:
    raise RuntimeError(f"Unexpected proxy health response: {proxy_status}")
reported_upstream = proxy_status.get("upstream")
if reported_upstream and reported_upstream.rstrip("/") != COMFY_URL:
    raise RuntimeError(f"The API proxy points to {reported_upstream}, not {COMFY_URL}.")

APP_ENV = os.environ.copy()
APP_ENV.update({
    "COMFY_BASE_URL": PROXY_URL,
    "COMFY_UI_URL": COMFY_URL,
    "PORT": "7860",
    "HOST": "127.0.0.1",
    "ACCESS_KEY": ACCESS_KEY,
})
APP_ENV.pop("COMFY_API_KEY", None)
run(["npm", "run", "check-env"], cwd=APP_DIR, env=APP_ENV)

## 9. Launch the app and open it

This starts the production build on port 7860 and creates one temporary HTTPS tunnel. Open the printed link; the generated access key is exchanged for a session cookie.

In [ ]:
app_log = BASE_DIR / "h3-motion-app.log"
app_process = PROCESSES.get("app")
try:
    app_status = read_json(
        f"{APP_URL}/api/health", headers={"Cookie": f"h3m_access={ACCESS_KEY}"}
    )
except Exception:
    if app_process is not None and app_process.poll() is None:
        raise RuntimeError(f"H3 Motion is running but not responding.\n{log_tail(app_log)}")
    handle = app_log.open("w")
    LOG_HANDLES["app"] = handle
    PROCESSES["app"] = subprocess.Popen(
        ["npm", "start"],
        cwd=APP_DIR,
        env=APP_ENV,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    app_status = wait_for_json(
        f"{APP_URL}/api/health", "H3 Motion app", timeout=120,
        process=PROCESSES["app"], log_path=app_log,
        headers={"Cookie": f"h3m_access={ACCESS_KEY}"},
    )
else:
    if app_process is None or app_process.poll() is not None:
        raise RuntimeError(
            "Port 7860 is already serving an app outside this notebook. "
            "Use a fresh Colab runtime so this tunnel exposes the H3 Motion app."
        )
    print("Reusing this notebook's H3 Motion app.")

if app_status.get("server") != "ok" or app_status.get("comfy") != "ok":
    raise RuntimeError(f"App or ComfyUI health check failed: {app_status}")

if PROCESSES.get("tunnel") is None or PROCESSES["tunnel"].poll() is not None:
    tunnel_log = BASE_DIR / "cloudflared.log"
    handle = tunnel_log.open("w")
    LOG_HANDLES["tunnel"] = handle
    PROCESSES["tunnel"] = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", APP_URL],
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    deadline = time.monotonic() + 90
    match = None
    while time.monotonic() < deadline:
        if PROCESSES["tunnel"].poll() is not None:
            raise RuntimeError(f"Cloudflare Tunnel exited.\n{log_tail(tunnel_log)}")
        log_text = tunnel_log.read_text(errors="replace") if tunnel_log.exists() else ""
        match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            break
        time.sleep(1)
    if not match:
        raise TimeoutError(f"Cloudflare URL did not appear.\n{log_tail(tunnel_log)}")
    PUBLIC_URL = match.group(0)
else:
    tunnel_log = BASE_DIR / "cloudflared.log"
    log_text = tunnel_log.read_text(errors="replace") if tunnel_log.exists() else ""
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        PUBLIC_URL = match.group(0)

if not PUBLIC_URL:
    raise RuntimeError("The tunnel is running but its public URL could not be read.")

from IPython.display import Markdown, display

display(Markdown(f"### [Open H3 Motion]({PUBLIC_URL}/?key={ACCESS_KEY})"))
print("ComfyUI and its API proxy remain local; only the H3 Motion app is tunneled.")

## Stop the services

Run this cell when you are done. It stops only the processes started by this notebook.

In [ ]:
for name, process in reversed(list(PROCESSES.items())):
    if process is None or process.poll() is not None:
        continue
    print(f"Stopping {name}.")
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=10)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait(timeout=5)

PROCESSES.clear()
for handle in LOG_HANDLES.values():
    try:
        handle.close()
    except Exception:
        pass
LOG_HANDLES.clear()
PUBLIC_URL = None
print("H3 Motion, proxy, ComfyUI, and tunnel processes started by this notebook are stopped.")